# 우리 AI 만들기 (Colab 무료 GPU)

위에서부터 차례로 ▶ 를 누르면 됩니다. 1~2시간 걸려요.

0. **런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 고르세요.
1. 준비물: GitHub Actions의 **Make training data** 실행 결과 페이지 아래 Artifacts의 `lecture-notes-training` (zip)을 내려받아 두세요.
   (또는 직접 모은 데이터: `server/scripts/export_training.py`로 만든 `train.jsonl`, `val.jsonl`)

In [ ]:
# 1) 학습 도구 설치 (3~5분)
!pip install -q unsloth

In [ ]:
# 2) 데이터 올리기: lecture-notes-training.zip 을 고르세요 (또는 train.jsonl·val.jsonl·finetune.py·Modelfile 을 함께)
from google.colab import files
import os, zipfile
for name in files.upload():
    if name.endswith('.zip'):
        zipfile.ZipFile(name).extractall('.')
print(sorted(f for f in os.listdir('.') if not f.startswith('.')))
!wc -l train.jsonl val.jsonl

In [ ]:
# 3) 가르치기: 큰 AI의 답(사용자가 고친 것 반영)을 작은 모델 Qwen3-4B가 따라 하도록 (T4에서 데이터 수백 개 기준 30분~1시간)
# 실시간 필기만 먼저: --only annotate 를 붙이세요
!python finetune.py --train train.jsonl --val val.jsonl

In [ ]:
# 4) 결과 내려받기: Ollama용 모델 파일(약 2.5GB). 크면 Google Drive에 복사하는 편이 빠릅니다
import glob, shutil
gguf = glob.glob('lecture-notes-gguf/*.gguf')
print(gguf)
shutil.copy('Modelfile', 'lecture-notes-gguf/Modelfile')
files.download(gguf[0])
files.download('Modelfile')

In [ ]:
# (선택) Google Drive에 저장
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/lecture-notes && cp lecture-notes-gguf/*.gguf Modelfile /content/drive/MyDrive/lecture-notes/

## 내 컴퓨터에서 쓰기

```bash
# Ollama 설치: https://ollama.com
# Modelfile의 FROM 줄을 내려받은 .gguf 파일 이름으로 바꾼 뒤
ollama create lecture-notes -f Modelfile

cd server
LLM_MODE=ollama OLLAMA_MODEL=lecture-notes python scripts/eval_model.py   # 큰 AI와 비교 채점
LLM_MODE=ollama OLLAMA_MODEL=lecture-notes uvicorn app.main:app --port 8000
```